# 03 - Manuscript figures (journal-ready)

Rebuilds the two manuscript maps from `outputs/tables/excess_mortality.csv`.
No titles or footnotes are embedded in the images: captions live in the manuscript
(*Public Health* guide: figures on separate sheets, numbered, with legends).
Saves 300-dpi TIFF for submission (`manuscript/submission_figures/`) and PNG for the repo (`outputs/figures/`).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REF = f'{ROOT}/data/ref'; TAB = f'{ROOT}/outputs/tables'
PNG = f'{ROOT}/outputs/figures'; SUB = f'{ROOT}/manuscript/submission_figures'
os.makedirs(SUB, exist_ok=True)

df = pd.read_csv(f'{TAB}/excess_mortality.csv', dtype={'CODMUNRES': str})
df['sig_excess'] = df['sig_excess'].astype(str).eq('True')

gdf = gpd.read_file(f'{REF}/br_municipios.geojson')
gdf['CODMUNRES'] = gdf['id'].str[:6]
gdf = gdf.merge(df, on='CODMUNRES', how='left')
states = gpd.read_file(f'{REF}/br_states.geojson')
print(len(gdf), 'municipalities;', gdf.deaths_total.notna().sum(), 'with data')

## Figure 1 - choropleth of the empirical-Bayes SMR (Model A)

Grey = fewer than 50 accumulated deaths (not reliably typed). Caption in the manuscript.

In [ ]:
norm = TwoSlopeNorm(vmin=0.7, vcenter=1.0, vmax=1.5)
CMAP = 'RdYlGn_r'

fig, ax = plt.subplots(figsize=(11, 8.5))
reliable = gdf[gdf.deaths_total >= 50]
grey = gdf[~(gdf.deaths_total >= 50)]
grey.plot(ax=ax, color='#d9d9d9', edgecolor='white', linewidth=0.05)
reliable.plot(ax=ax, column='SMR_A_eb', cmap=CMAP, norm=norm,
              edgecolor='white', linewidth=0.05)
states.plot(ax=ax, facecolor='none', edgecolor='#555555', linewidth=0.5)

cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=CMAP), ax=ax,
                    extend='max', shrink=0.72, pad=0.02)
cbar.set_ticks([0.8, 1.0, 1.2, 1.4])
cbar.set_ticklabels(['0.8', '1.0\n(as expected)', '1.2', '1.4'])
cbar.set_label('Observed / expected deaths (empirical-Bayes SMR)', fontsize=11)

ax.legend(handles=[Patch(facecolor='#d9d9d9', edgecolor='#999999',
                          label='< 50 deaths (not reliably typed)')],
          loc='lower left', frameon=False, fontsize=10)
ax.set_axis_off()
fig.tight_layout()
fig.savefig(f'{SUB}/Figure1.tiff', dpi=300, bbox_inches='tight',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(f'{PNG}/fig1_excess_map.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2 - bubble map of the absolute avoidable burden

Bubble area = deaths above the structural expectation (reliable municipalities).
Dark red = significant excess (CI lower bound > 1). Caption in the manuscript.

In [ ]:
SIG, NONSIG = '#b03a48', '#f5b98f'
K = 0.6  # marker points^2 per excess death

bub = gdf[(gdf.deaths_total >= 50) & (gdf.excess_deaths > 0)].copy()
bub['pt'] = bub.geometry.representative_point()

fig, ax = plt.subplots(figsize=(10, 10))
states.plot(ax=ax, facecolor='#f2f2f2', edgecolor='#999999', linewidth=0.6)
for sig, colour, z in [(False, NONSIG, 2), (True, SIG, 3)]:
    part = bub[bub.sig_excess == sig]
    ax.scatter(part.pt.x, part.pt.y, s=part.excess_deaths * K,
               c=colour, alpha=0.65, edgecolors='none', zorder=z)

size_handles = [ax.scatter([], [], s=v * K, c='#9a9a9a', alpha=0.8,
                           label=f'{v} excess deaths') for v in (50, 250, 1000)]
leg1 = ax.legend(handles=size_handles, title='Bubble = deaths above expected',
                 loc='lower left', frameon=False, labelspacing=2.0,
                 borderpad=1.0, fontsize=10, title_fontsize=10)
ax.add_artist(leg1)
colour_handles = [ax.scatter([], [], s=110, c=SIG, label='significant excess (CI > 1)'),
                  ax.scatter([], [], s=110, c=NONSIG, label='excess, not significant')]
ax.legend(handles=colour_handles, loc='lower right', frameon=False, fontsize=10)
ax.set_axis_off()
fig.tight_layout()
fig.savefig(f'{SUB}/Figure2.tiff', dpi=300, bbox_inches='tight',
            pil_kwargs={'compression': 'tiff_lzw'})
fig.savefig(f'{PNG}/fig2_burden_map.png', dpi=300, bbox_inches='tight')
plt.show()